# Extracting ITS_LIVE velocity data 

## Packages to install (needs to be done via conda activate in terminal):
- zarr 
- pyproj
- s3fs

## In terminal:
The version needs to be specified so the versions don't conflict with each other, run this in a new terminal (in VS code), once the packages have been installed in conda:
"pip install "zarr<3" "fsspec==2025.3.0" s3fs xarray pyproj" 

## Script info: 
This notebook is in here for reference, but it may not run in VSCode. In order to extract the velocity maps, it needed AWS access which requires admin approval. I got around it by running in Colab since their server already has access. Additionally, if you do want to run it in colab, it is extremely slow, it took about 4 hours to finsih running. 

## Other data: 
- I emailed you guys my tidal data set, the code below is set up for it to like in the data folder of our project folder. We can dicuss where to host it later. Right now, those files are in the git ignore 

In [6]:
import geopandas as gpd
import pandas as pd
import numpy as np
import xarray as xr
import rioxarray as rxr
import matplotlib.pyplot as plt
from functools import partial

from scipy.signal import savgol_filter as sg
from tqdm import tqdm

from shapely.geometry import Polygon
from shapely.geometry import Point
from scipy import ndimage
import shapely.geometry
import imageio.v2 as imageio

import os
import dask
from dask.delayed import delayed

from blue_ice_tools_simple import get_urls
from blue_ice_tools_simple import get_data_cube 

GPS_stations = [(-138.0800,-83.5200), ( -138.7500,-83.4650), ( -137.4000, -83.6200), ( -138.3500, -83.5800),( -139.3000,-83.4100),
    (  -139.1000, -83.6700),
    ( -137.8000, -83.5500),
    ( -140.0000,-83.4900),
    (  -137.7200, -83.4200),
    ( -138.1700,  -83.6380,)]

print(get_urls(points=GPS_stations))


['https://its-live-data.s3.amazonaws.com/datacubes/v2-updated-october2024/S80W140/ITS_LIVE_vel_EPSG3031_G0120_X-450000_Y-550000.zarr']


In [6]:
get_urls(points=GPS_stations)
get_data_cube(urls=get_urls(points=GPS_stations), start_date="2008-01-01", end_date="2019-12-31")

/home/k-sanchez126/miniconda3/envs/GPGN268/lib/python3.14/site-packages/dask/_task_spec.py:767: FutureWarning: In a future version, xarray will not decode the variable 'date_dt' into a timedelta64 dtype based on the presence of a timedelta-like 'units' attribute by default. Instead it will rely on the presence of a timedelta64 'dtype' attribute, which is now xarray's default way of encoding timedelta64 values.
To continue decoding into a timedelta64 dtype, either set `decode_timedelta=True` when opening this dataset, or add the attribute `dtype='timedelta64[ns]'` to this variable on disk.
To opt-in to future behavior, set `decode_timedelta=False`.
  return self.func(*new_argspec, **kwargs)


<xarray.Dataset> Size: 200MB
Dimensions:   (mid_date: 36, y: 833, x: 833)
Coordinates:
  * mid_date  (mid_date) datetime64[ns] 288B 2017-01-31 ... 2019-12-31
  * y         (y) float64 7kB -5.001e+05 -5.002e+05 ... -5.998e+05 -5.999e+05
  * x         (x) float64 7kB -4.999e+05 -4.997e+05 ... -4.001e+05 -4e+05
Data variables:
    vx        (mid_date, y, x) float32 100MB dask.array<chunksize=(36, 470, 470), meta=np.ndarray>
    vy        (mid_date, y, x) float32 100MB dask.array<chunksize=(36, 470, 470), meta=np.ndarray>
Attributes: (12/19)
    Conventions:                CF-1.8
    GDAL_AREA_OR_POINT:         Area
    author:                     ITS_LIVE, a NASA MEaSUREs project (its-live.j...
    autoRIFT_parameter_file:    http://its-live-data.s3.amazonaws.com/autorif...
    datacube_software_version:  1.0
    date_created:               26-Sep-2023 10:12:20
    ...                         ...
    s3:                         s3://its-live-data/datacubes/v2-updated-octob...
    skipped_granules:           s3://its-live-data/datacubes/v2-updated-octob...
    time_standard_img1:         UTC
    time_standard_img2:         UTC
    title:                      ITS_LIVE datacube of image pair velocities
    url:                        https://its-live-data.s3.amazonaws.com/datacu...

In [4]:
# Check what URLs get_urls actually returns for your stations
urls = get_urls(points=GPS_stations)
print(urls)

/mnt/c/Users/kaitl/OneDrive/Desktop/work/classes/GPGN268/SP2026-FP01-Ice-Sheet-Velocity/Notebooks/blue_ice_tools_simple.py:1198: SyntaxWarning: "\D" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\D"? A raw string is also an option.
  '$\Delta \sigma_{VM}$ [kPa]',
/mnt/c/Users/kaitl/OneDrive/Desktop/work/classes/GPGN268/SP2026-FP01-Ice-Sheet-Velocity/Notebooks/blue_ice_tools_simple.py:1199: SyntaxWarning: "\D" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\D"? A raw string is also an option.
  '$\Delta \sigma_{1}$ [kPa]',
/mnt/c/Users/kaitl/OneDrive/Desktop/work/classes/GPGN268/SP2026-FP01-Ice-Sheet-Velocity/Notebooks/blue_ice_tools_simple.py:1200: SyntaxWarning: "\D" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\D"? A raw string is also an option.
  '$\Delta \sigma_{2}$ [kPa]'


TypeError: Points must be a list, tuple, or None.

In [11]:
print(type(GPS_stations[0]))
print(GPS_stations[0])

<class 'tuple'>
(-138.08, -83.52)


In [5]:
# Check what URLs get_urls actually returns for your stations
urls = get_urls(points=GPS_stations)
print(urls)

TypeError: Points must be a list, tuple, or None.

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import xarray as xr
import rioxarray as rxr
import matplotlib.pyplot as plt
from functools import partial

from scipy.signal import savgol_filter as sg
from tqdm import tqdm

from shapely.geometry import Polygon
from shapely.geometry import Point
from scipy import ndimage
import shapely.geometry
import imageio.v2 as imageio

import os
import dask
from dask.delayed import delayed

from blue_ice_tools_simple import get_urls
from blue_ice_tools_simple import get_data_cube 

GPS_stations = [(-138.0800,-83.5200), ( -138.7500,-83.4650), ( -137.4000, -83.6200), ( -138.3500, -83.5800),( -139.3000,-83.4100),
    (  -139.1000, -83.6700),
    ( -137.8000, -83.5500),
    ( -140.0000,-83.4900),
    (  -137.7200, -83.4200),
    ( -138.1700,  -83.6380,)]

#Load your existing features CSV 
df = pd.read_csv("../Data/Whillians-GPS-Data-and-Features.csv", parse_dates=["start_time"])

# Pull the datacube 
urls = get_urls(points=GPS_stations)
dc   = get_data_cube(urls=urls, start_date="2008-01-01", end_date="2019-12-31")
# dc is an xr.Dataset with dims (mid_date, y, x)

# For every GPS station, extract vx/vy time-series 
# GPS_stations should be a list/dict of {"name": ..., "lat": ..., "lon": ...}
# (or whatever structure get_urls() expects — adjust as needed)

station_records = []

for station in GPS_stations:
    name = station["name"]         
    lat  = station["lat"]
    lon  = station["lon"]

    # Select the nearest grid cell to this station
    # ITS_LIVE datacubes are in EPSG:3031 (polar stereographic),
    # so you may need to reproject lat/lon  x/y first:
    transformer = pyproj.Transformer.from_crs("EPSG:4326", "EPSG:3031", always_xy=True)
    px, py = transformer.transform(lon, lat)

    pt = dc.sel(x=px, y=py, method="nearest")   # shape: (mid_date,)

    vx_series = pt["vx"].values   # numpy array, length = n_timesteps
    vy_series = pt["vy"].values
    dates      = pd.to_datetime(pt["mid_date"].values)

    tmp = pd.DataFrame({
        "mid_date":    dates,
        f"vx_{name}":  vx_series,
        f"vy_{name}":  vy_series,
    })
    station_records.append(tmp)

# Combine all station velocity series into one wide DataFrame 
# Each station gets its own vx_<name> / vy_<name> column, aligned on mid_date
from functools import reduce
vel_df = reduce(lambda a, b: pd.merge(a, b, on="mid_date", how="outer"),
                station_records)

# Merge with your features CSV on the nearest mid_date 
# ITS_LIVE mid_date is the centre of the image-pair window (~12–48 day spacing),
# so we do a nearest-neighbour temporal join rather than an exact match.
df   = df.sort_values("start_time").reset_index(drop=True)
vel_df = vel_df.sort_values("mid_date").reset_index(drop=True)

merged = pd.merge_asof(
    df,
    vel_df,
    left_on  = "start_time",
    right_on = "mid_date",
    direction = "nearest",       # picks the closest mid_date for every row
    tolerance = pd.Timedelta("30D"),  # drop match if > 30 days away (tune as needed)
)

# Inspect & save 
print(merged.shape)
print(merged.head())
merged.to_csv("Whillians-GPS-Data-and-Features-with-velocity.csv", index=False)

/mnt/c/Users/kaitl/OneDrive/Desktop/work/classes/GPGN268/SP2026-FP01-Ice-Sheet-Velocity/Notebooks/blue_ice_tools_simple.py:1198: SyntaxWarning: "\D" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\D"? A raw string is also an option.
  '$\Delta \sigma_{VM}$ [kPa]',
/mnt/c/Users/kaitl/OneDrive/Desktop/work/classes/GPGN268/SP2026-FP01-Ice-Sheet-Velocity/Notebooks/blue_ice_tools_simple.py:1199: SyntaxWarning: "\D" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\D"? A raw string is also an option.
  '$\Delta \sigma_{1}$ [kPa]',
/mnt/c/Users/kaitl/OneDrive/Desktop/work/classes/GPGN268/SP2026-FP01-Ice-Sheet-Velocity/Notebooks/blue_ice_tools_simple.py:1200: SyntaxWarning: "\D" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\D"? A raw string is also an option.
  '$\Delta \sigma_{2}$ [kPa]'


In [ ]:
import pandas as pd
import numpy as np
import xarray as xr
import s3fs
import zarr
from pyproj import Transformer

# Config 
GPS_stations = {
    "WB07": {"lat": -83.5200, "lon": -138.0800},
    "WB08": {"lat": -83.4650, "lon": -138.7500},
    "WB09": {"lat": -83.6200, "lon": -137.4000},
    "WB10": {"lat": -83.5800, "lon": -138.3500},
    "WB11": {"lat": -83.4100, "lon": -139.3000},
    "WB12": {"lat": -83.6700, "lon": -139.1000},
    "WB13": {"lat": -83.5500, "lon": -137.8000},
    "WB14": {"lat": -83.4900, "lon": -140.0000},
    "ENB":  {"lat": -83.4200, "lon": -137.7200},
    "SLW":  {"lat": -83.6380, "lon": -138.1700},
}

# Load events CSV 
df = pd.read_csv("../Data/Whillians-GPS-Data-and-Features.csv", parse_dates=["start_time"])
df = df.sort_values("start_time").reset_index(drop=True)
print(f"Catalog: {len(df):,} events  {df['start_time'].min().date()} → {df['start_time'].max().date()}")

# Open datacube 
ZARR_S3 = "s3://its-live-data/datacubes/v2/S80W140/ITS_LIVE_vel_EPSG3031_G0120_X-450000_Y-550000.zarr"
fs    = s3fs.S3FileSystem(anon=True)
store = s3fs.S3Map(root=ZARR_S3, s3=fs, check=False)
ds    = xr.open_zarr(store, consolidated=True, decode_timedelta=False)
ds    = ds.sortby("mid_date")
print(f"Datacube: {str(ds.mid_date.values[0])[:10]} → {str(ds.mid_date.values[-1])[:10]}")

# Project station coords 
tr = Transformer.from_crs("EPSG:4326", "EPSG:3031", always_xy=True)
station_xy = {
    name: tr.transform(coords["lon"], coords["lat"])
    for name, coords in GPS_stations.items()
}

# Extract full time series per station (one S3 fetch each)
vel_frames = []

for station, (sx, sy) in station_xy.items():
    print(f"  Fetching {station}...")
    ds_pt = ds[["v", "vx", "vy"]].sel(x=sx, y=sy, method="nearest").compute()  # one fetch

    vel_frames.append(pd.DataFrame({
        "mid_date":        pd.to_datetime(ds_pt["mid_date"].values),
        f"v_myr_{station}":  ds_pt["v"].values.astype(float),
        f"vx_myr_{station}": ds_pt["vx"].values.astype(float),
        f"vy_myr_{station}": ds_pt["vy"].values.astype(float),
    }))

# Combine all stations into one wide velocity DataFrame 
from functools import reduce
vel_df = reduce(lambda a, b: pd.merge(a, b, on="mid_date", how="outer"), vel_frames)
vel_df = vel_df.sort_values("mid_date").reset_index(drop=True)

# Save the raw velocity data 
vel_df.to_csv("whillans_velocity_timeseries.csv", index=False)
print(f"Velocity table: {vel_df.shape}  saved to whillans_velocity_timeseries.csv")

# Nearest-date merge into events catalog 
merged = pd.merge_asof(
    df,
    vel_df,
    left_on   = "start_time",
    right_on  = "mid_date",
    direction = "nearest",
    tolerance = pd.Timedelta("30D"),
)

print(f"\nDone: {len(merged):,} rows, {merged.shape[1]} columns")
print(merged[[c for c in merged.columns if "v_myr" in c]].describe())

# Save 
merged.to_csv("whillans_velocity_slip_merged.csv", index=False)
print("Saved whillans_velocity_slip_merged.csv")

Catalog: 5,150 events  2008-01-25 → 2019-11-23
Datacube: 2017-01-08 → 2022-01-27
  Fetching WB07...
  Fetching WB08...
  Fetching WB09...
  Fetching WB10...
  Fetching WB11...
  Fetching WB12...
  Fetching WB13...
  Fetching WB14...
  Fetching ENB...
  Fetching SLW...
Velocity table: (132, 31)  saved to whillans_velocity_timeseries.csv


MergeError: incompatible merge keys [0] dtype('<M8[us]') and dtype('<M8[ns]'), must be the same type

In [ ]:

#urls = get_urls(points=GPS_stations)
ZARR_S3 = urls[0].replace("https://its-live-data.s3.amazonaws.com/", "s3://its-live-data/")

fs    = s3fs.S3FileSystem(anon=True)
store = s3fs.S3Map(root=ZARR_S3, s3=fs, check=False)
ds    = xr.open_zarr(store, consolidated=True, decode_timedelta=False)
ds    = ds.sortby("mid_date")

print("First date:", str(ds.mid_date.values[0])[:10])
print("Last date: ", str(ds.mid_date.values[-1])[:10])
print("Total epochs:", len(ds.mid_date))
print("Shape:", ds.dims)


tr = Transformer.from_crs("EPSG:4326", "EPSG:3031", always_xy=True)
px, py = tr.transform(-138.08, -83.52)
pt = ds.sel(x=px, y=py, method="nearest")
print("\nStation point date range:")
print("First:", str(pt.mid_date.values[0])[:10])
print("Last: ", str(pt.mid_date.values[-1])[:10])
print("Non-NaN vx count:", int((~np.isnan(pt["vx"].values)).sum()))

First date: 2017-01-08
Last date:  2025-01-13
Total epochs: 150
Shape: FrozenMappingWarningOnValuesAccess({'mid_date': 150, 'y': 833, 'x': 833})

Station point date range:
First: 2017-01-08
Last:  2025-01-13
Non-NaN vx count: 11


In [ ]:
# Check if v2  has the same tile and what dates it covers
print("v2 S80W140 files:")
print(fs.ls("its-live-data/datacubes/v2/S80W140/"))

import json
import pandas as pd
from pyproj import Transformer

# Load the datacube catalog
with fs.open("its-live-data/datacubes/catalog_v02.json") as f:
    catalog = json.load(f)

# Project station coords to EPSG:3031
tr = Transformer.from_crs("EPSG:4326", "EPSG:3031", always_xy=True)

GPS_stations = {
    "WB07": {"lat": -83.5200, "lon": -138.0800},
    "SLW":  {"lat": -83.6380, "lon": -138.1700},
}

for name, coords in GPS_stations.items():
    px, py = tr.transform(coords["lon"], coords["lat"])
    print(f"\n{name} → EPSG:3031: x={px:.0f}, y={py:.0f}")

# Print all catalog entries for S80W140 region
print("\nCatalog entries for S80W140:")
for entry in catalog.get("features", []):
    props = entry.get("properties", {})
    url = props.get("zarr_url", "")
    if "S80W140" in url:
        print(url, "→", props.get("date_start","?"), "to", props.get("date_end","?"))

v2 S80W140 files:
['its-live-data/datacubes/v2/S80W140/ITS_LIVE_vel_EPSG3031_G0120_X-350000_Y-450000.json', 'its-live-data/datacubes/v2/S80W140/ITS_LIVE_vel_EPSG3031_G0120_X-350000_Y-450000.zarr', 'its-live-data/datacubes/v2/S80W140/ITS_LIVE_vel_EPSG3031_G0120_X-350000_Y-550000.json', 'its-live-data/datacubes/v2/S80W140/ITS_LIVE_vel_EPSG3031_G0120_X-350000_Y-550000.zarr', 'its-live-data/datacubes/v2/S80W140/ITS_LIVE_vel_EPSG3031_G0120_X-450000_Y-550000.json', 'its-live-data/datacubes/v2/S80W140/ITS_LIVE_vel_EPSG3031_G0120_X-450000_Y-550000.zarr', 'its-live-data/datacubes/v2/S80W140/ITS_LIVE_vel_EPSG3031_G0120_X-450000_Y-650000.json', 'its-live-data/datacubes/v2/S80W140/ITS_LIVE_vel_EPSG3031_G0120_X-450000_Y-650000.zarr', 'its-live-data/datacubes/v2/S80W140/ITS_LIVE_vel_EPSG3031_G0120_X-450000_Y-750000.json', 'its-live-data/datacubes/v2/S80W140/ITS_LIVE_vel_EPSG3031_G0120_X-450000_Y-750000.zarr', 'its-live-data/datacubes/v2/S80W140/ITS_LIVE_vel_EPSG3031_G0120_X-550000_Y-750000.json', 'i

In [18]:
for f in sorted(os.listdir("../Data/Events/")):
    print(f)

import os

# Peek inside one .evt file to see what's in it
evt_path = "../Data/Events/2008_2008Events2stas/2008-02-06_15-49-30.evt"
with open(evt_path, "r") as f:
    print(f.read())

2008_2008Events2stas
2009_2009Events2stas
2010_2010Events2stas
2011_2011Events2stas
2012_2012Events2stas
2013_2013Events2stas
2014_2014Events2stas
2015_2015Events2stas
2016_2016Events2stas
2017_2017Events2stas
2018_2018Events2stas
2019_2019Events2stas
	time	la09x	la09y	la09z	la09res	la09res_avg	slw1x	slw1y	slw1z	slw1res	slw1res_avg	sum_res_avg	ressum	event
211120	2008-02-06 15:49:30	-291076.12474095315	-504843.19758754846	108.2957	0.017273518692841692	0.012358728334929557	-278674.12077481434	-560056.924395824	92.1984	0.07526280945906523	0.06566145107381831	0.07802017940874786	0.09253632815190692	1.0
211121	2008-02-06 15:49:45	-291076.1239265249			0.017532807346227745	0.012358728334929557	-278674.11446899315	-560056.9204370108	92.2194	0.07526280945906523	0.06566145107381831	0.07802017940874786	0.09279561680529297	1.0
211122	2008-02-06 15:50:00	-291076.1231120967	-504843.1953846391	108.2908	0.017792095999613802	0.012358728334929557	-278674.1179464594	-560056.9180308328	92.2075	0.07526280